A Docker image is not a monolithic file you copy around. It is a **stack of read-only layers**, each representing a change to the filesystem — and that layered design is why Docker images are so fast to build, so efficient to store, and so cheap to share between machines.

In this notebook we'll cover:
- What an image actually is on disk
- How the layer system works and why it exists
- The overlay2 storage driver — the engine that merges layers
- Copy-on-Write: how containers get their own writable layer
- Image manifests, configs, and content-addressable storage
- Tags, digests, and how images are identified

## What is a Docker Image?

A Docker image is a **read-only, ordered stack of filesystem layers** plus a small JSON config that records metadata (environment variables, working directory, default command, exposed ports, and the layer order).

It is *not* a single file like a VM disk image. On disk, each layer is stored independently as a tar archive of filesystem changes. Docker composes them into one coherent filesystem at runtime using a union filesystem.

The analogy that fits best: **a Git history**.

| Git | Docker image |
|---|---|
| Each commit records only the *diff* — what changed | Each layer records only the filesystem changes made by one build instruction |
| History is a stack of diffs applied in order | Layers are stacked in order to produce the final filesystem |
| Two branches can share a common ancestor commit | Two images can share base layers — stored only once on disk |
| A commit is identified by a SHA-256 hash | A layer is identified by a SHA-256 digest |

When you `docker pull python:3.12-slim`, Docker downloads each layer separately, checks its digest, and stores it once — even if ten different images on your machine share that same base layer.

## How Layers Are Built

Every instruction in a Dockerfile that modifies the filesystem creates a new layer. Instructions that only set metadata (`ENV`, `EXPOSE`, `LABEL`, `CMD`, `ENTRYPOINT`) create zero-size metadata layers.

```dockerfile
FROM ubuntu:22.04          # Layer 1: unpack ubuntu:22.04 base (~30 MB)
RUN apt-get update && \    # Layer 2: filesystem changes from apt-get (~50 MB)
    apt-get install -y curl
COPY app.py /app/          # Layer 3: adds /app/app.py (~1 KB)
ENV PORT=8080              # metadata only — no filesystem change
CMD ["python", "/app/app.py"]  # metadata only
```

The result is a stack of three filesystem layers:

```
┌─────────────────────────────┐  ← Layer 3: /app/app.py added
├─────────────────────────────┤  ← Layer 2: /usr/bin/curl, apt cache, etc.
├─────────────────────────────┤  ← Layer 1: full Ubuntu 22.04 filesystem
└─────────────────────────────┘  (base)
```

**Layer caching** is the key performance feature. When Docker builds an image, it hashes the instruction and its inputs. If the hash matches a cached layer, Docker reuses it instantly — no re-execution. This is why instruction order in a Dockerfile matters so much: put rarely-changing instructions (like `apt-get install`) early and frequently-changing instructions (like `COPY . .`) late, so the cache stays valid as long as possible.

## The overlay2 Storage Driver

Docker needs a way to present multiple read-only layers as a single coherent filesystem to a running container. The default mechanism on Linux is the **overlay2** storage driver, which uses the Linux kernel's **OverlayFS**.

OverlayFS merges two directory trees:

```
lowerdir  = the image layers (read-only, stacked)
upperdir  = the container's writable layer (read-write)
workdir   = scratch space OverlayFS needs internally
─────────────────────────────────────────
merged    = what the container sees (the union of all of the above)
```

From inside the container, you see one unified filesystem. You can read any file from any image layer. When you write or modify a file, it is placed in `upperdir` — the image layers beneath are never touched.

```
Container's view (merged):
  /etc/nginx/nginx.conf   ← comes from lowerdir (image layer)
  /var/log/nginx/         ← comes from upperdir (container writes)
  /app/config.json        ← modified: copy is in upperdir, original in lowerdir
```

On disk, the layers live under `/var/lib/docker/overlay2/` on Linux. Each layer is a directory containing only the files that changed in that layer. The kernel does the merging transparently.

## Copy-on-Write

**Copy-on-Write (CoW)** is the mechanism that lets hundreds of containers share the same image layers without interfering with each other.

When a container process reads a file that exists in a lower image layer, it reads it directly — no copying needed. When a container process *writes to* or *modifies* a file that exists in a lower layer, OverlayFS:

1. Copies the file from the image layer up into the container's `upperdir`
2. The write is applied to the copy in `upperdir`
3. On the next read, the container sees the modified copy in `upperdir` — the original in the image layer is unchanged

```
100 containers sharing nginx:alpine image
─────────────────────────────────────────
Image layers (read-only, shared):    ~25 MB × 1  =  ~25 MB on disk
Container writable layers (upperdir): ~0 MB × 100 = near 0 MB (until writes happen)

Without CoW:
Full copy per container:             ~25 MB × 100 = ~2.5 GB on disk
```

**Important:** the writable layer (upperdir) is **ephemeral**. It exists only as long as the container exists. When you `docker rm` a container, its writable layer is deleted. If you want data to survive past a container's lifetime, use a **volume** or **bind mount** (covered in a later topic).

## Image Manifest and Content-Addressable Storage

When Docker pulls an image from a registry, it downloads two types of artifact:

**The manifest** — a JSON document that lists the layers that make up the image and references the image config:

```json
{
  "schemaVersion": 2,
  "config": {
    "digest": "sha256:abc123...",
    "size": 4621
  },
  "layers": [
    { "digest": "sha256:def456...", "size": 3166745 },
    { "digest": "sha256:ghi789...", "size": 1234567 }
  ]
}
```

**The config** — a JSON document that records the image's metadata: environment variables, working directory, exposed ports, the command to run, and the layer history.

Everything is stored by its **SHA-256 digest** — this is content-addressable storage. If two images share a layer with the same digest, they literally share the same bytes on disk. Docker never downloads a layer it already has, regardless of which image it belongs to.

A **manifest list** (also called a multi-platform manifest) is a layer above individual manifests. It points to different image manifests for different CPU architectures (`amd64`, `arm64`, `arm/v7`). When you `docker pull ubuntu`, Docker automatically selects the manifest for your machine's architecture.

## Tags and Digests

Images are referred to using the pattern `[registry/][namespace/]repository[:tag][@digest]`.

| Reference | Meaning |
|---|---|
| `nginx` | `docker.io/library/nginx:latest` — registry, namespace, and tag all defaulted |
| `nginx:1.25` | Specific tag — `1.25` is a mutable pointer; the registry owner can update it |
| `nginx:1.25-alpine` | Tag that describes both version and base OS variant |
| `nginx@sha256:abc…` | Immutable digest reference — always the exact same bytes, forever |
| `ghcr.io/myorg/api:v2.1` | Image on GitHub Container Registry |

**Tags are mutable.** When you pull `python:3.12-slim` today and again in three months, you may get different bytes — the tag has been updated by the Python maintainers with security patches. This is desirable for staying current, but it means builds are not automatically reproducible.

**Digests are immutable.** `python@sha256:d4e5f6…` will always refer to exactly that version of the image. CI/CD pipelines that need reproducible builds pin to digests.

**Choosing a tag:**

| Tag style | Example | When to use |
|---|---|---|
| `latest` | `node:latest` | Never in production — unpredictable |
| Major version | `node:20` | Accepts minor + patch updates automatically |
| Full version | `node:20.11.1` | Predictable; update explicitly |
| Slim / Alpine | `node:20-slim` | Smaller image; fewer pre-installed tools |
| Digest | `node@sha256:…` | Fully reproducible; use in production Dockerfiles |

## Hands-On: Exploring Image Layers

The cells below pull images and inspect their layer structure.

In [ ]:
# Pull python:3.12-slim and watch each layer download separately
# Each 'Pull complete' line is one image layer
!docker pull python:3.12-slim

In [ ]:
# Show the layer history — each row is one Dockerfile instruction
# SIZE = how much that layer adds to the image; 0B = metadata-only instruction
!docker history python:3.12-slim

In [ ]:
# Pull a second image that shares the same Debian base as python:3.12-slim
# Shared layers print 'Already exists' — no download, no extra disk usage
!docker pull python:3.11-slim

In [ ]:
# List both images — despite both being full Python installs, shared layers
# mean the combined disk footprint is much less than the sum of their sizes
!docker image ls python --format 'table {{.Repository}}\t{{.Tag}}\t{{.Size}}'

In [ ]:
# Inspect the image manifest — shows the layer digests and sizes as Docker sees them
import subprocess, json

raw = subprocess.check_output(["docker", "inspect", "python:3.12-slim"])
data = json.loads(raw)[0]

print("ID (short)  :", data["Id"][:19])
print("Architecture:", data["Architecture"])
print("OS          :", data["Os"])
print("Size        :", f"{data['Size'] / 1_000_000:.1f} MB")
print("Layers      :", len(data["RootFS"]["Layers"]))
print()
print("Layer digests:")
for i, layer in enumerate(data["RootFS"]["Layers"], 1):
    print(f"  {i}: {layer[:32]}…")

In [ ]:
# Build a tiny image from scratch to see layers accumulate
# We write a minimal Dockerfile inline and build it

dockerfile = """\
FROM alpine:3.19
RUN echo 'layer 2: installing curl' && apk add --no-cache curl
RUN echo 'layer 3: creating a file' && echo hello > /greeting.txt
LABEL maintainer="learning"
CMD ["cat", "/greeting.txt"]
"""

with open("/tmp/Dockerfile.demo", "w") as f:
    f.write(dockerfile)

!docker build -f /tmp/Dockerfile.demo -t layers-demo:v1 /tmp
print()
!docker history layers-demo:v1

In [ ]:
# Demonstrate the layer cache — rebuild without changes
# Every step says 'CACHED' — Docker reuses all layers from the first build
!docker build -f /tmp/Dockerfile.demo -t layers-demo:v1 /tmp

In [ ]:
# Show that a running container has a thin writable layer on top
# We write a file inside the container — the image layers are unchanged
!docker run --rm layers-demo:v1 sh -c "\
    echo 'writing to container layer...' && \
    echo 'this lives in upperdir' > /runtime-file.txt && \
    ls /"

# After the container exits, /runtime-file.txt is gone — the writable layer was discarded
print("\nRunning a second container from the same image — no runtime-file.txt:")
!docker run --rm layers-demo:v1 ls /

In [ ]:
# Show the image's immutable digest — pin this in production Dockerfiles
!docker inspect --format '{{index .RepoDigests 0}}' python:3.12-slim 2>/dev/null || \
 docker inspect --format '{{.Id}}' python:3.12-slim

In [ ]:
# Clean up the demo image
!docker rmi layers-demo:v1

## Summary

- A Docker image is a **read-only stack of filesystem layers** plus a JSON config — not a monolithic file.
- Each Dockerfile instruction that modifies the filesystem creates a **new layer** containing only the diff. Metadata instructions (`ENV`, `CMD`, `EXPOSE`) create zero-size layers.
- **Layer caching** reuses unchanged layers at build time — keep infrequently-changing instructions early in the Dockerfile to maximise cache hits.
- The **overlay2** storage driver uses Linux OverlayFS to merge read-only image layers into a single filesystem view (`lowerdir`) plus a writable container layer (`upperdir`).
- **Copy-on-Write** means reads come directly from image layers (fast, shared), and writes copy the file into the container's `upperdir` first — image layers are never modified.
- The container's **writable layer is ephemeral** — it disappears when the container is removed. Use volumes for persistent data.
- Layers are stored by **SHA-256 digest** (content-addressable). Shared layers between images are stored once on disk and never re-downloaded.
- **Tags are mutable**; **digests are immutable**. Pin to digests for reproducible production builds.